# 02 — Explore Lagrangian Particle Trajectories

This notebook demonstrates how to load and visualize **Lagrangian particle trajectories** (~200,000 synthetic tracer particles) from the CylinderWake3900 dataset.

**Unique feature**: This dataset provides *paired* Lagrangian and Eulerian data, enabling data assimilation and Lagrangian-to-Eulerian reconstruction tasks.

**Paper**: [Khojasteh et al. (2021), Data in Brief](https://doi.org/10.1016/j.dib.2021.107725)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from cylinderwake import CylinderWake3900
from cylinderwake.visualize import plot_trajectories

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Load Lagrangian Data

In [ ]:
ds = CylinderWake3900(field='lagrangian', subdomain='near', download=True)

print(f'Number of snapshots: {len(ds)}')

sample = ds[0]
print(f"Positions shape:  {sample['positions'].shape}")   # (N_particles, 3)
print(f"Velocities shape: {sample['velocities'].shape}")  # (N_particles, 3)
print(f"Time: {sample['time']}")

## 2. Particle Scatter Plot (Single Snapshot)

In [ ]:
pos = sample['positions'].numpy() if hasattr(sample['positions'], 'numpy') else sample['positions']
vel = sample['velocities'].numpy() if hasattr(sample['velocities'], 'numpy') else sample['velocities']

fig = plot_trajectories(
    pos, vel,
    max_particles=5000,
    color_by='velocity',
    s=2, alpha=0.5,
    title='Particle positions colored by velocity magnitude',
)
plt.savefig('lagrangian_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. 3D Scatter Plot

In [ ]:
fig = plot_trajectories(
    pos, vel,
    max_particles=2000,
    color_by='velocity',
    projection='3d',
    s=2, alpha=0.3,
    title='3D particle distribution',
)
plt.show()

## 4. Full Particle Trajectories Over Time

In [ ]:
# Load all time steps and reconstruct trajectories
trajectories = ds.get_full_trajectories()

print(f"Positions shape:  {trajectories['positions'].shape}")   # (N_times, N_particles, 3)
print(f"Velocities shape: {trajectories['velocities'].shape}")
print(f"Number of time steps: {len(trajectories['times'])}")

In [ ]:
pos_all = trajectories['positions']
if hasattr(pos_all, 'numpy'):
    pos_all = pos_all.numpy()

fig = plot_trajectories(
    pos_all,
    max_particles=200,
    alpha=0.2,
    title='Lagrangian particle trajectories (x-y projection)',
)
plt.savefig('lagrangian_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Particle Velocity Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

labels = ['$u_p$', '$v_p$', '$w_p$']
for i in range(3):
    axes[i].hist(vel[:, i], bins=100, density=True, alpha=0.7, color='steelblue')
    axes[i].set_xlabel(labels[i])
    axes[i].set_ylabel('PDF')
    axes[i].set_title(f'Particle velocity {labels[i]}')

plt.suptitle('Lagrangian velocity PDF (single snapshot)', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Sparse Observation Subsampling

For data assimilation benchmarks: subsample particles to simulate sparse PTV observations.

In [ ]:
# Simulate sparse observations: keep only 1% of particles
n_total = pos.shape[0]
sparsity_levels = [1.0, 0.1, 0.01, 0.001]

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, frac in zip(axes, sparsity_levels):
    n_keep = max(1, int(n_total * frac))
    idx = np.random.choice(n_total, n_keep, replace=False)
    speed = np.linalg.norm(vel[idx], axis=1)
    ax.scatter(pos[idx, 0], pos[idx, 1], c=speed, s=3, alpha=0.6, cmap='viridis')
    ax.set_title(f'{frac*100:.1f}% ({n_keep} particles)')
    ax.set_aspect('equal')
    ax.set_xlabel('x / D')

axes[0].set_ylabel('y / D')
plt.suptitle('Sparse Lagrangian observations (data assimilation benchmark)', fontsize=13)
plt.tight_layout()
plt.savefig('sparse_observations.png', dpi=150, bbox_inches='tight')
plt.show()

---

**Next**: See `03_baseline_forecasting.ipynb` for a simple ML baseline using this data.